# Hello

## 1. Load Data

In [ ]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [ ]:
# Dataset Metadata
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

In [ ]:
# Display users
import glob
import os

users = sorted([
    p for p in Path(DATA_DOWNLOAD_DIR, "emg2pose_dataset_mini").iterdir()
    if p.is_dir()
])

for i, x in enumerate(users):
    print(f"{i}: {x}")

In [ ]:
# Display sessions for chosen user

user = users[0] # choose a user

sessions = sorted(glob.glob(os.path.join(user, "*.hdf5")))
for i, x in enumerate(sessions):
    print(f"{i}: {x}")

## 2. Select Session To Evaluate On

In [ ]:
from emg2pose.data import Emg2PoseSessionData

# Select our dataset 
session = sessions[0]
print(session)
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
# View the data
print(data.fields)
print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

In [ ]:
from emg2pose.utils import downsample

joint_angles_30hz = downsample(data['emg'], native_fs=2000, target_fs=30)
joint_angles_30hz.shape

In [ ]:
# Show our sessions metadata

metadata_df[metadata_df["filename"] == data.metadata["filename"]]

In [ ]:
# Downscale ground-truth joint angles and display it over time

import emg2pose.visualization as visualization
from emg2pose.utils import downsample
import numpy as np

joint_angles = data["joint_angles"]
joint_angles_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250])

In [ ]:
# Show where inverse kinematics tracking (ground truth labeling)
visualization.ik_failure_plot(data)

## 3. Generate Predictions Of This Session
Load the EMG data in to the models

### Select Session(s) to Train On
Certain methods require training our own model using various ML methods.
We use our defined `sessions` list that are avaialbe to train on
and set start and end indeces to decide what subset of `sessions` to train on.

In [ ]:
print(sessions)

print(len(sessions))

In [ ]:
# --- parameters ---

start_idx = 1
end_idx = len(sessions)  # inclusive

# start_idx = 0
# end_idx = 0  # inclusive

train = sessions[start_idx:end_idx+1]
train

### 3a. Load Checkpoint and Predict
Loads a large pretrained model from Meta Reality labs and runs it on the data

In [ ]:
from pathlib import Path
import os

checkpoint_dir = Path(DATA_DOWNLOAD_DIR) / "emg2pose_model_checkpoints"

# Download checkpoint if it does not exist
if not checkpoint_dir.exists():
    os.system(f'''
    cd {DATA_DOWNLOAD_DIR} &&
    curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_model_checkpoints.tar.gz" -o emg2pose_model_checkpoints.tar.gz &&
    tar -xvzf emg2pose_model_checkpoints.tar.gz
    ''')

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DATA_DOWNLOAD_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

In [ ]:
from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
import torch

session = data 
start_idx = 0
stop_idx = len(session["emg"]) # eval on the whole session

session_window = session[start_idx:stop_idx]

# no_ik_failure is not a field so we slice separately
no_ik_failure_window = session.no_ik_failure[start_idx:stop_idx]

batch = {
    "emg": torch.Tensor([session_window["emg"].T]),  # BCT
    "joint_angles": torch.Tensor([session_window["joint_angles"].T]),  # BCT
    "no_ik_failure": torch.Tensor([no_ik_failure_window]),  # BT
}

preds, joint_angles, no_ik_failure = module.forward(batch)

# Algorithms that use the initial state for ground truth will do poorly
# when the first joint angles are missing!
if (joint_angles[:, 0] == 0).all():
    print(
        "Warning! Ground truth not available at first time step!"
    )

# BCT --> TC (as numpy)
preds = preds[0].T.detach().numpy()
joint_angles = joint_angles[0].T.detach().numpy()

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Ensure alignment and scale

print(preds.shape)
print(joint_angles.shape)

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

print(preds_30hz.shape)
print(ground_truth_30hz.shape)

mask = no_ik_failure.cpu().numpy().squeeze()  # -> (T,)
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()  # ensure (T,)

preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

print(preds_30hz.shape)
print(ground_truth_30hz.shape)


In [ ]:
from emg2pose.evaluation import evaluate_predictions

evaluate_predictions(preds_30hz, ground_truth_30hz)

### 3b Small LSTM

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- load multiple sessions (RAW EMG) ---
X_all, y_all = [], []

for s in train:
    data = Emg2PoseSessionData(hdf5_path=s)
    X_all.append(data['emg'])
    y_all.append(data['joint_angles'])

X = np.concatenate(X_all, axis=0)
y = np.concatenate(y_all, axis=0)

print("Raw X shape:", X.shape)

# --- BETTER downsampling ---
ds_factor = 4   # 2000 Hz → 500 Hz (NOT too aggressive)
X = X[::ds_factor]
y = y[::ds_factor]

print("Downsampled X shape:", X.shape)

# --- sequence params ---
seq_len = 100   # more context (~200 ms at 500 Hz)
target_samples = 20_000

# --- adaptive stride ---
raw_N = len(X) - seq_len
stride = max(1, raw_N // target_samples)
print(f"Using stride={stride}")

# --- sequence building ---
def make_sequences(X, y, seq_len, stride):
    Xs, ys = [], []
    for i in range(0, len(X) - seq_len, stride):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len - 1])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X, y, seq_len, stride)

print("Sequence X shape:", X_seq.shape)

# --- split ---
split_idx = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

# --- tensors ---
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

# --- BETTER Conv + LSTM model ---
class BetterConvLSTM(nn.Module):
    def __init__(self, in_ch, out_dim):
        super().__init__()

        # keep temporal resolution (no stride)
        self.conv = nn.Conv1d(
            in_channels=in_ch,
            out_channels=32,
            kernel_size=9,
            stride=1,
            padding=4
        )

        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = x.transpose(1, 2)      # (B, C, T)
        x = self.conv(x)           # (B, 32, T)
        x = self.relu(x)
        x = x.transpose(1, 2)      # (B, T, 32)

        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = BetterConvLSTM(X.shape[1], y.shape[1])

# --- train ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}: loss = {loss.item():.4f}")

# --- eval ---
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb))

y_pred = torch.cat(preds).numpy()
y_test = y_test.numpy()

In [ ]:
print(y.shape)

print(y_test.shape)
print(y_pred.shape)

In [ ]:
Emg2PoseSessionData(hdf5_path=train[0])

In [ ]:
# --- inference on RAW EMG (MATCHES training pipeline) ---

def predict_stream(model, X_raw, y_raw, seq_len=100, ds_factor=4, stride=1):
    model.eval()
    preds = []

    # --- SAME preprocessing as training ---
    X = X_raw[::ds_factor]
    y = y_raw[::ds_factor]

    for i in range(0, len(X) - seq_len, stride):
        window = X[i:i+seq_len]
        window = torch.tensor(window[None, ...], dtype=torch.float32)

        with torch.no_grad():
            pred = model(window).cpu().numpy()

        preds.append(pred[0])

    preds = np.array(preds)

    # --- GT aligned EXACTLY like training ---
    y_gt = []
    for i in range(0, len(y) - seq_len, stride):
        y_gt.append(y[i + seq_len - 1])

    y_gt = np.array(y_gt)

    return preds, y_gt


# --- usage ---
data = Emg2PoseSessionData(hdf5_path=train[0])

X_raw = data['emg']
y_raw = data['joint_angles']

y_pred, y_gt = predict_stream(
    model,
    X_raw,
    y_raw,
    seq_len=100,
    ds_factor=4,
    stride=1   # dense inference
)

In [ ]:
effective_hz = 500 / stride

gt_30hz = downsample(y_gt, effective_hz, 30)
preds_30hz = downsample(y_pred, effective_hz, 30)

print(gt_30hz.shape)
print(preds_30hz.shape)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
visualization.get_plotly_animation_for_joint_angles(ground_truth_30hz[0:250], color="pink")

In [ ]:
evaluate_predictions(preds_30hz, gt_30hz)

LSTM trained on user x achieved very good MAE when evaluated on that user.
Now we will consider this model trained on user x to see generalizabiltiy to other users (user y).

In [ ]:
user_y = sessions[30]
user_y

In [ ]:
# --- inference on raw EMG sequence ---

def predict_stream(model, X, seq_len=50):
    model.eval()
    preds = []

    for i in range(len(X) - seq_len):
        window = X[i:i+seq_len]
        window = torch.tensor(window[None, ...], dtype=torch.float32)

        with torch.no_grad():
            pred = model(window).cpu().numpy()

        preds.append(pred[0])

    return np.array(preds)


# example usage:
user_y_data = Emg2PoseSessionData(hdf5_path=user_y)
X_raw = user_y_data['emg']

y_pred = predict_stream(model, X_raw, seq_len=50)

In [ ]:
ground_truth = user_y_data['joint_angles']


In [ ]:
preds_30hz = downsample(y_pred, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
ground_truth_30hz = downsample(ground_truth, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(ground_truth_30hz[0:250], color="pink")

In [ ]:
import importlib
import emg2pose.evaluation
importlib.reload(emg2pose.evaluation)

from emg2pose.evaluation import evaluate_predictions

In [ ]:
evaluate_predictions(preds_30hz, ground_truth_30hz)

### 3c Meta Model with Limited Training Data

In [ ]:
from emg2pose.train_subset import train_subset

checkpoint = train_subset(train, DATA_DOWNLOAD_DIR)

In [ ]:
checkpoint

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={checkpoint}"
    ]
)

from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
import torch

session = data

start_idx = 0
stop_idx = len(session) # eval on the whole session

session_window = session[start_idx:stop_idx]

# no_ik_failure is not a field so we slice separately
no_ik_failure_window = session.no_ik_failure[start_idx:stop_idx]

batch = {
    "emg": torch.Tensor([session_window["emg"].T]),
    "joint_angles": torch.Tensor([session_window["joint_angles"].T]),
    "no_ik_failure": torch.Tensor([no_ik_failure_window]),
}

batch = {k: v.to(module.device) for k, v in batch.items()}

preds, joint_angles, no_ik_failure = module.forward(batch)

# Algorithms that use the initial state for ground truth will do poorly
# when the first joint angles are missing!
if (joint_angles[:, 0] == 0).all():
    print(
        "Warning! Ground truth not available at first time step!"
    )

# BCT --> TC (as numpy)
preds = preds[0].T.detach().cpu().numpy()
joint_angles = joint_angles[0].T.detach().cpu().numpy()

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Visualize the gt

ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250], color="pink")

In [ ]:
# Ensure alignment and scale

print(preds.shape)
print(joint_angles.shape)

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

print(preds_30hz.shape)
print(ground_truth_30hz.shape)

mask = no_ik_failure.cpu().numpy().squeeze()  # -> (T,)
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()  # ensure (T,)

preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

print(preds_30hz.shape)
print(ground_truth_30hz.shape)


In [ ]:
from emg2pose.evaluation import evaluate_predictions

evaluate_predictions(preds_30hz, ground_truth_30hz)

### 3d Classical Machine Learning

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
import numpy as np
from emg2pose.feature_extraction import features
import numpy as np

# --- build feature dataset from selected training sessions ---
X_all, y_all = [], []

for s in train:
    data = Emg2PoseSessionData(hdf5_path=s)
    X_feats, y_out = features(data)

    X_all.append(X_feats)
    y_all.append(y_out)

X = np.concatenate(X_all, axis=0)
y = np.concatenate(y_all, axis=0)

print("Feature X shape:", X.shape)
print("Target y shape:", y.shape)

# --- split (preserve time order) ---
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# --- RIDGE ---
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

print("Ridge done:", ridge_pred.shape)

# --- SVR ---
svr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", MultiOutputRegressor(
        SVR(kernel="rbf", C=1.0, epsilon=0.1)
    ))
])

svr_model.fit(X_train, y_train)
svr_pred = svr_model.predict(X_test)

print("SVR done:", svr_pred.shape)

# --- PLS  ---
pls = PLSRegression(n_components=10)  # try 5–20 range if needed
pls.fit(X_train, y_train)

pls_pred = pls.predict(X_train)

print("PLS done:", pls_pred.shape)

In [ ]:
# Ground Truth Visualization

visualization.get_plotly_animation_for_joint_angles(
    y_test[0:250],
    color="pink"
)

In [ ]:
# Ridge Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    ridge_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(ridge_pred, y_test)

In [ ]:
# SVR Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    svr_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(svr_pred, y_test)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(
    pls_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(pls_pred, y_test)